In [ ]:

import numpy as np
import pandas as pd
import rasterio
import os
import re
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# ======================================================================
# CONFIGURACIÓN GENERAL
# ======================================================================

tf.keras.backend.set_floatx('float32')

nx, ny = 450, 387        # (cols, rows)
dx, dy = 10.0, 10.0
dt = 60.0 * 5.0         # 15 min en segundos
SEQUENCE_LENGTH = 36      # longitud de secuencia para Conv1D

# --------- FIXED PATHS ----------
case_dir = os.path.normpath(r"../Data/caso2")
image_base_dir = os.path.join(case_dir, "water_depth_tiffs")   # .../caso1/water_depth_tiffs
hidrograma_base_dir = case_dir                                 # .../caso1
# --------------------------------

# ======================================================================
# FUNCIONES AUXILIARES
# ======================================================================

def load_tif_image(path, target_size=(ny, nx)):
    """
    Carga una imagen TIFF y la convierte en un array float32.
    Si falla, genera un array de ceros con el tamaño esperado.
    target_size debe ser (ny, nx) porque rasterio lee (rows, cols).
    """
    try:
        with rasterio.open(path) as src:
            img = src.read(1).astype(np.float32)
            if img.shape != target_size:
                img = np.resize(img, target_size)
            return img
    except Exception:
        return np.zeros(target_size, dtype=np.float32)


def _extract_scenario_id_from_filename(filepath):
    """
    Extrae el id numérico desde 'hidrograma_sintetico_{id}.txt'.
    Si no puede, retorna None.
    """
    base = os.path.basename(filepath)
    m = re.search(r"hidrograma_sintetico_(\d+)\.txt$", base)
    return int(m.group(1)) if m else None


def load_hidrograma(hidrograma_file):
    """
    Carga hidrograma desde archivo.
    Si no existe, genera uno sintético (mockup) en el mismo path.
    """
    if not os.path.exists(hidrograma_file):
        print(f"MOCKUP: Creando archivo {hidrograma_file} con datos sintéticos.")

        os.makedirs(os.path.dirname(hidrograma_file), exist_ok=True)

        N = 300
        tiempo = np.arange(N) * dt / 3600  # horas
        caudal = np.sin(tiempo * 0.1) * 10 + 50
        data = np.stack([tiempo, caudal], axis=1)

        # crear carpeta del escenario de imágenes (si necesitas que exista)
        os.makedirs(image_base_dir, exist_ok=True)
        scenario_id = _extract_scenario_id_from_filename(hidrograma_file)
        if scenario_id is not None:
            scenario_dir = os.path.join(image_base_dir, f"escenario_{scenario_id}")
            os.makedirs(scenario_dir, exist_ok=True)

        header = "Hidrograma sintético\nLinea2\nLinea3\nLinea4\n"
        with open(hidrograma_file, "w") as f:
            f.write(header)
            np.savetxt(f, data, fmt="%.4f")

        return data

    return np.loadtxt(hidrograma_file, skiprows=4)

def interpolate_hidrograma(hidrograma, dt_target):
    t = hidrograma[:, 0]
    q = hidrograma[:, 1]

    # Nuevo eje temporal
    t_new = np.arange(t[0], t[-1] + dt_target, dt_target)

    # Interpolación
    q_new = np.interp(t_new, t, q)

    return np.column_stack([t_new, q_new])
# ======================================================================
# GENERADOR → MATRICES X_local Y_local
# ======================================================================

def compute_volume_from_flat(y_flat, dx, dy):
    """
    Calcula el volumen total a partir de una imagen de profundidades flatten.
    """
    return np.sum(y_flat) * dx * dy

def generate_dataset_with_mass_terms(image_dir, hidrograma_values):
    """
    Genera:
    - X_local: (n, SEQUENCE_LENGTH)
    - Y_local: (n, nx*ny)
    - V_prev, V_next: (n,)
    - Q_t, Q_next: (n,)
    """
    try:
        images = sorted([f for f in os.listdir(image_dir)
                         if f.lower().endswith((".tif", ".tiff"))])
    except FileNotFoundError:
        images = [f"mock_img_{i}.tif" for i in range(len(hidrograma_values) + 2)]

    start_idx = SEQUENCE_LENGTH - 1
    end_idx = min(len(hidrograma_values) - 2, len(images) - 2)

    X_local, Y_local = [], []
    V_prev, V_next = [], []
    Q_t, Q_next = [], []
    
    for i in range(start_idx, end_idx):

        # -----------------
        # Inputs (Q window)
        # -----------------
        x_seq = hidrograma_values[i - SEQUENCE_LENGTH + 1:i + 1, 1]#.astype(np.float32)
        #print(x_seq)

        # -----------------
        # Depth fields
        # -----------------
        img_prev = load_tif_image(os.path.join(image_dir, images[i]),
                                  target_size=(ny, nx))
        img_next = load_tif_image(os.path.join(image_dir, images[i + 1]),
                                  target_size=(ny, nx))
        img_next2 = load_tif_image(os.path.join(image_dir, images[i + 2]),
                                   target_size=(ny, nx))

        y_flat = img_next.flatten().astype(np.float32)

        # -----------------
        # Volumes
        # -----------------
        v_prev = compute_volume_from_flat(img_prev, dx, dy)
        v_next = compute_volume_from_flat(img_next2, dx, dy)

        # -----------------
        # Discharges
        # -----------------
        q_t = hidrograma_values[i, 1]
        q_next = hidrograma_values[i + 1, 1]

        # -----------------
        # Append
        # -----------------
        X_local.append(x_seq)
        Y_local.append(y_flat)
        V_prev.append(v_prev)
        V_next.append(v_next)
        Q_t.append(q_t)
        Q_next.append(q_next)

    return (
        np.array(X_local, dtype=np.float32),
        np.array(Y_local, dtype=np.float32),
        np.array(V_prev, dtype=np.float32),
        np.array(V_next, dtype=np.float32),
        np.array(Q_t, dtype=np.float32),
        np.array(Q_next, dtype=np.float32),
    )

# ======================================================================
# CONSTRUCCIÓN GLOBAL: Big_X y Big_Y
# ======================================================================

# -----------------------------
# Choose scenario splits
# -----------------------------
train_scenarios = [i for i in range(0, 43)]
test_scenarios = [7,14,21,28,35,40]
val_scenarios   = [5, 18, 32]   # ← tú decides

train_scenarios = [
    i for i in train_scenarios
    if i not in test_scenarios + val_scenarios
]

# -----------------------------
# Build per-scenario datasets
# -----------------------------
scenario_data = {}

for scenario_id in range(43):
    print(f"\nProcesando escenario {scenario_id}")

    image_dir = os.path.join(image_base_dir, f"escenario_{scenario_id}")
    hidrograma_file = os.path.join(
        hidrograma_base_dir,
        f"hidrograma_sintetico_{scenario_id}.txt"
    )

    hidrograma_raw = load_hidrograma(hidrograma_file)
    hidrograma_values = interpolate_hidrograma(hidrograma_raw, dt_target=300)
    
    (
        X_local,
        Y_local,
        V_prev_local,
        V_next_local,
        Q_t_local,
        Q_next_local,
    ) = generate_dataset_with_mass_terms(image_dir, hidrograma_values)

    print(f"  → Samples escenario {scenario_id}: {X_local.shape[0]}")

    scenario_data[scenario_id] = (
        X_local,
        Y_local,
        V_prev_local,
        V_next_local,
        Q_t_local,
        Q_next_local,
    )


def concat_scenarios(scenario_ids, scenario_data_dict):
    X_list, Y_list = [], []
    Vp_list, Vn_list = [], []
    Qt_list, Qn_list = [], []

    for sid in scenario_ids:
        data = scenario_data_dict[sid]
        if data[0].size == 0:
            continue

        Xs, Ys, Vp, Vn, Qt, Qn = data

        X_list.append(Xs)
        Y_list.append(Ys)
        Vp_list.append(Vp)
        Vn_list.append(Vn)
        Qt_list.append(Qt)
        Qn_list.append(Qn)

    return (
        np.vstack(X_list).astype(np.float32),
        np.vstack(Y_list).astype(np.float32),
        np.concatenate(Vp_list).astype(np.float32),
        np.concatenate(Vn_list).astype(np.float32),
        np.concatenate(Qt_list).astype(np.float32),
        np.concatenate(Qn_list).astype(np.float32),
    )

(
    X_train,
    Y_train,
    V_prev_train,
    V_next_train,
    Q_t_train,
    Q_next_train,
) = concat_scenarios(train_scenarios, scenario_data)

(
    X_val,
    Y_val,
    V_prev_val,
    V_next_val,
    Q_t_val,
    Q_next_val,
) = concat_scenarios(val_scenarios, scenario_data)

(
    X_test,
    Y_test,
    V_prev_test,
    V_next_test,
    Q_t_test,
    Q_next_test,
) = concat_scenarios(test_scenarios, scenario_data)

eps = 1e-8

x_min = np.min(X_train)
x_max = np.max(X_train)

X_train_n = (X_train - x_min) / (x_max - x_min + eps)
X_val_n   = (X_val   - x_min) / (x_max - x_min + eps)
X_test_n  = (X_test  - x_min) / (x_max - x_min + eps)

print("Normalization OK")
print("X_train:", X_train_n.shape)
print("X_val  :", X_val_n.shape)
print("X_test :", X_test_n.shape)


H0_MASK = 0.01  # metros (ajústalo si tu raster usa otra convención)
Y_train_mask = (Y_train > H0_MASK).astype(np.float32)
Y_val_mask   = (Y_val   > H0_MASK).astype(np.float32)
Y_test_mask  = (Y_test  > H0_MASK).astype(np.float32)


def depth_loss_flat(y_true, y_pred, h0=H0_MASK, wet_weight=4.0, dry_penalty=2.0):
    wet = tf.cast(y_true > h0, tf.float32)
    w = 1.0 + wet_weight * wet
    base = tf.reduce_sum(w * tf.square(y_true - y_pred)) / (tf.reduce_sum(w) + 1e-8)

    #dry = tf.cast(y_true <= h0, tf.float32)
    #spurious = tf.nn.relu(y_pred - h0) * dry
    #dry_term = tf.reduce_sum(spurious) / (tf.reduce_sum(dry) + 1e-8)

    return base #+ dry_penalty * dry_term
def weighted_bce(y_true, y_pred, pos_weight=20.0, eps=1e-7):
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    # BCE per-pixel
    bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    # weight positives
    w = 1.0 + (pos_weight - 1.0) * y_true
    return tf.reduce_mean(w * bce)

def dice_loss(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0-1e-7)

    # flatten per-sample
    y_true_f = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    y_pred_f = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])

    inter = tf.reduce_sum(y_true_f * y_pred_f, axis=1)
    denom = tf.reduce_sum(y_true_f + y_pred_f, axis=1)
    dice = (2.0 * inter + smooth) / (denom + smooth)
    return 1.0 - tf.reduce_mean(dice)

def mask_loss_combo(y_true, y_pred, pos_weight=20.0, dice_w=1.0, bce_w=1.0):
    return bce_w * weighted_bce(y_true, y_pred, pos_weight=pos_weight) + dice_w * dice_loss(y_true, y_pred)

# 1. Define the Custom Model Class to handle extra physics inputs
class AblationFloodModel(models.Model):
    def __init__(self, cnn_model, dx, dy, dt, nx, ny, use_physics=True, use_mask=True):
        super(AblationFloodModel, self).__init__()
        self.cnn_model = cnn_model
        self.dx, self.dy, self.dt = tf.cast(dx, tf.float32), tf.cast(dy, tf.float32), tf.cast(dt, tf.float32)
        self.nx, self.ny = tf.cast(nx, tf.float32), tf.cast(ny, tf.float32)
        self.area = self.dx * self.dy * self.nx * self.ny
        self.use_physics = use_physics
        self.use_mask = use_mask

    def call(self, inputs):
        return self.cnn_model(inputs)
    
    def train_step(self, data):
        x, target_dict = data
        y_true_depth = target_dict["depth_flat"]
        
        with tf.GradientTape() as tape:
            y_pred_depth, y_pred_mask = self.cnn_model(x, training=True)
            
            # 1. Depth Loss (Always Active)
            loss_depth = depth_loss_flat(y_true_depth, y_pred_depth)
            
            # 2. Mask Loss (Conditional)
            loss_mask = tf.constant(0.0)
            if self.use_mask:
                y_true_mask = target_dict["wetmask_flat"]
                loss_mask = mask_loss_combo(y_true_mask, y_pred_mask, pos_weight=20.0, dice_w=1.0, bce_w=1.0)

            # 3. Physics Loss (Conditional)
            loss_physics = tf.constant(0.0)
            if self.use_physics:
                v_prev, v_next = target_dict["v_prev"], target_dict["v_next"]
                q_t, q_next = target_dict["q_t"], target_dict["q_next"]
                v_hat = tf.reduce_sum(y_pred_depth, axis=1) * self.dx * self.dy
                reg1 = tf.reduce_mean(tf.square(tf.nn.relu(v_hat - v_prev - self.dt * q_t) / self.area))
                reg2 = tf.reduce_mean(tf.square(tf.nn.relu(v_next - v_hat - self.dt * q_next) / self.area))
                loss_physics = reg1 + reg2

            total_loss = loss_depth + (1.0 * loss_mask) + (10.0 * loss_physics)

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        return {
            "loss": total_loss, 
            "depth_loss": loss_depth, 
            "mask_loss": loss_mask, 
            "phys_loss": loss_physics
        }

    def test_step(self, data):
        x, target_dict = data
        y_true_depth = target_dict["depth_flat"]
        y_pred_depth, _ = self.cnn_model(x, training=False)
        loss_depth = depth_loss_flat(y_true_depth, y_pred_depth)
        return {"loss": loss_depth, "depth_loss": loss_depth}



2026-02-17 09:50:00.395145: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-17 09:50:00.773142: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 09:50:02.261732: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.



Procesando escenario 0
  → Samples escenario 0: 252

Procesando escenario 1
  → Samples escenario 1: 252

Procesando escenario 2
  → Samples escenario 2: 252

Procesando escenario 3
  → Samples escenario 3: 252

Procesando escenario 4
  → Samples escenario 4: 252

Procesando escenario 5
  → Samples escenario 5: 252

Procesando escenario 6
  → Samples escenario 6: 252

Procesando escenario 7
  → Samples escenario 7: 252

Procesando escenario 8
  → Samples escenario 8: 252

Procesando escenario 9
  → Samples escenario 9: 252

Procesando escenario 10
  → Samples escenario 10: 252

Procesando escenario 11
  → Samples escenario 11: 252

Procesando escenario 12
  → Samples escenario 12: 252

Procesando escenario 13
  → Samples escenario 13: 252

Procesando escenario 14
  → Samples escenario 14: 252

Procesando escenario 15
  → Samples escenario 15: 252

Procesando escenario 16
  → Samples escenario 16: 252

Procesando escenario 17
  → Samples escenario 17: 252

Procesando escenario 18
  → S

In [2]:
H0_MASK = 0.01  # metros (ajústalo si tu raster usa otra convención)
Y_train_mask = (Y_train > H0_MASK).astype(np.float32)
Y_val_mask   = (Y_val   > H0_MASK).astype(np.float32)
Y_test_mask  = (Y_test  > H0_MASK).astype(np.float32)

In [7]:
class AblationFloodModel(models.Model):
    def __init__(self, cnn_model, dx, dy, dt, nx, ny, use_physics=True, use_mask=True):
        super(AblationFloodModel, self).__init__()
        self.cnn_model = cnn_model
        self.dx, self.dy, self.dt = tf.cast(dx, tf.float32), tf.cast(dy, tf.float32), tf.cast(dt, tf.float32)
        self.nx, self.ny = tf.cast(nx, tf.float32), tf.cast(ny, tf.float32)
        self.area = self.dx * self.dy * self.nx * self.ny
        self.use_physics = use_physics
        self.use_mask = use_mask

    def call(self, inputs):
        return self.cnn_model(inputs)
    
    def train_step(self, data):
        x, target_dict = data
        y_true_depth = target_dict["depth_flat"]
        
        with tf.GradientTape() as tape:
            y_pred_depth, y_pred_mask = self.cnn_model(x, training=True)
            
            # 1. Depth Loss (Always Active)
            loss_depth = depth_loss_flat(y_true_depth, y_pred_depth)
            
            # 2. Mask Loss (Conditional)
            loss_mask = tf.constant(0.0)
            if self.use_mask:
                y_true_mask = target_dict["wetmask_flat"]
                loss_mask = mask_loss_combo(y_true_mask, y_pred_mask, pos_weight=20.0, dice_w=1.0, bce_w=1.0)

            # 3. Physics Loss (Conditional)
            loss_physics = tf.constant(0.0)
            if self.use_physics:
                v_prev, v_next = target_dict["v_prev"], target_dict["v_next"]
                q_t, q_next = target_dict["q_t"], target_dict["q_next"]
                v_hat = tf.reduce_sum(y_pred_depth, axis=1) * self.dx * self.dy
                reg1 = tf.reduce_mean(tf.square(tf.nn.relu(v_hat - v_prev - self.dt * q_t) / self.area))
                reg2 = tf.reduce_mean(tf.square(tf.nn.relu(v_next - v_hat - self.dt * q_next) / self.area))
                loss_physics = reg1 + reg2

            total_loss = loss_depth + (1.0 * loss_mask) + (10.0 * loss_physics)

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        return {
            "loss": total_loss, 
            "depth_loss": loss_depth, 
            "mask_loss": loss_mask, 
            "phys_loss": loss_physics
        }

    def test_step(self, data):
        x, target_dict = data
        y_true_depth = target_dict["depth_flat"]
        y_pred_depth, _ = self.cnn_model(x, training=False)
        loss_depth = depth_loss_flat(y_true_depth, y_pred_depth)
        return {"loss": loss_depth, "depth_loss": loss_depth}

In [8]:
def build_expert(
    sequence_length,
    output_size,
    hidden_channels=32,
    fc_units=[64, 32],
    dropout_rate=0.2
):
    inp = layers.Input(shape=(sequence_length, 1))

    x = layers.Conv1D(hidden_channels, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv1D(hidden_channels * 2, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv1D(hidden_channels * 4, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Flatten()(x)

    for units in fc_units:
        x = layers.Dense(units, activation="relu")(x)
        x = layers.Dropout(dropout_rate)(x)

    depth = layers.Dense(output_size, activation=None)(x)
    wetmask = layers.Dense(output_size, activation="sigmoid")(x)

    return models.Model(inp, [depth, wetmask])

def build_gate(inp, n_experts):
    x = layers.Conv1D(16, 3, padding="same", activation="relu")(inp)
    x = layers.Conv1D(16, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Bidirectional(
        layers.GRU(16, return_sequences=False)
    )(x)

    x = layers.Dense(32, activation="relu")(x)
    return layers.Dense(n_experts, activation="softmax", name="regime_gate")(x)

def build_discontinuous_flood_moe(
    output_size,
    sequence_length,
    n_experts=4,
    h0=0.01,
    alpha=80.0
):
    # Entrada: ventana de caudal
    inp = layers.Input(shape=(sequence_length, 1), name="Q_window")

    # ---------------------------
    # GATE (decide régimen)
    # ---------------------------
    gate = build_gate(inp, n_experts)   # 👈 AQUÍ SE USA
    
    # ---------------------------
    # Expertos independientes
    # ---------------------------
    experts = [
        build_expert(sequence_length, output_size)
        for _ in range(n_experts)
    ]

    expert_outputs = [exp(inp) for exp in experts]
    depths = [out[0] for out in expert_outputs]
    wetmasks = [out[1] for out in expert_outputs]

    # ---------------------------
    # Combinación ponderada
    # ---------------------------
    def weighted_sum(tensors):
        values, weights = tensors
        weights = tf.expand_dims(weights, axis=-1)
        return tf.reduce_sum(values * weights, axis=1)

    depth_stack = layers.Lambda(lambda x: tf.stack(x, axis=1))(depths)
    wetmask_stack = layers.Lambda(lambda x: tf.stack(x, axis=1))(wetmasks)

    depth = layers.Lambda(weighted_sum, name="depth_flat")([depth_stack, gate])
    wetmask = layers.Lambda(weighted_sum, name="wetmask_flat")([wetmask_stack, gate])

    model = models.Model(inp, [depth, wetmask])
    return model

def load_ablation_model(weights_path, nx, ny, dx, dy, dt, sequence_length, n_exp, use_physics=True, use_mask=True):
    # 1. Rebuild the MoE architecture
    base_cnn_new = build_discontinuous_flood_moe(
        output_size=nx * ny,
        sequence_length=sequence_length,
        n_experts=n_exp
    )
    
    # 2. Load the weights into the CNN
    # Note: We load weights into base_cnn_new because that's where the parameters live
    base_cnn_new.load_weights(weights_path)
    
    # 3. Wrap in the Ablation class with the correct flags
    loaded_model = AblationFloodModel(
        base_cnn_new,
        dx, dy, dt, nx, ny,
        use_physics=use_physics,
        use_mask=use_mask
    )
    
    # Compile so it's ready for .evaluate() or .predict()
    loaded_model.compile(optimizer=Adam(1e-3))
    
    print(f"✅ Model loaded: {weights_path} (Experts: {n_exp}, Phys: {use_physics}, Mask: {use_mask})")
    return loaded_model

# Ejemplo de uso:
# 1. Baseline
model_baseline_loaded = load_ablation_model(
    "ablation_models/checkpoints_model_baseline/model_baseline.weights.h5", 
    nx, ny, dx, dy, dt, SEQUENCE_LENGTH, n_exp=1, use_physics=False, use_mask=False
)

# 2. MoE Only
model_moe_only_loaded = load_ablation_model(
    "ablation_models/checkpoints_model_moe_only/model_moe_only.weights.h5", 
    nx, ny, dx, dy, dt, SEQUENCE_LENGTH, n_exp=4, use_physics=False, use_mask=False
)

# 3. Mask Single
model_mask_single_loaded = load_ablation_model(
    "ablation_models/checkpoints_model_mask_single/model_mask_single.weights.h5", 
    nx, ny, dx, dy, dt, SEQUENCE_LENGTH, n_exp=1, use_physics=False, use_mask=True
)

# 4. Mask MoE
model_mask_moe_loaded = load_ablation_model(
    "ablation_models/checkpoints_model_mask_moe/model_mask_moe.weights.h5", 
    nx, ny, dx, dy, dt, SEQUENCE_LENGTH, n_exp=4, use_physics=False, use_mask=True
)

# 5. PINN Single
model_pinn_single_loaded = load_ablation_model(
    "ablation_models/checkpoints_model_pinn_single/model_pinn_single.weights.h5", 
    nx, ny, dx, dy, dt, SEQUENCE_LENGTH, n_exp=1, use_physics=True, use_mask=False
)

# 6. PINN MoE Full
model_full_loaded = load_ablation_model(
    "ablation_models/checkpoints_model_full/model_full.weights.h5", 
    nx, ny, dx, dy, dt, SEQUENCE_LENGTH, n_exp=4, use_physics=True, use_mask=True
)


✅ Model loaded: ablation_models/checkpoints_model_baseline/model_baseline.weights.h5 (Experts: 1, Phys: False, Mask: False)
✅ Model loaded: ablation_models/checkpoints_model_moe_only/model_moe_only.weights.h5 (Experts: 4, Phys: False, Mask: False)
✅ Model loaded: ablation_models/checkpoints_model_mask_single/model_mask_single.weights.h5 (Experts: 1, Phys: False, Mask: True)
✅ Model loaded: ablation_models/checkpoints_model_mask_moe/model_mask_moe.weights.h5 (Experts: 4, Phys: False, Mask: True)
✅ Model loaded: ablation_models/checkpoints_model_pinn_single/model_pinn_single.weights.h5 (Experts: 1, Phys: True, Mask: False)
✅ Model loaded: ablation_models/checkpoints_model_full/model_full.weights.h5 (Experts: 4, Phys: True, Mask: True)


In [13]:
def depth_loss_flat_np(y_true, y_pred, h0=0.01, wet_weight=4.0, eps=1e-8):
    """
    Replica tu idea: weighted MSE (más peso donde y_true>h0).
    y_true, y_pred: (N, P)
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    wet = (y_true > h0).astype(np.float32)
    w = 1.0 + wet_weight * wet
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w) + eps
    return float(num / den)

def csi_np(y_true_bin, y_pred_bin, eps=1e-8):
    """
    CSI global (a nivel de pixeles) por escenario.
    y_true_bin, y_pred_bin: (N, P) boolean
    """
    y_true_bin = np.asarray(y_true_bin, dtype=bool)
    y_pred_bin = np.asarray(y_pred_bin, dtype=bool)

    tp = np.sum(y_true_bin & y_pred_bin)
    fp = np.sum(~y_true_bin & y_pred_bin)
    fn = np.sum(y_true_bin & ~y_pred_bin)
    return float(tp / (tp + fp + fn + eps))

def summarize_ablation_scenarios(
    scenario_ids,
    models_dict, # Recibe un diccionario { "nombre_modelo": objeto_modelo }
    dt_target,
    h0,
    sequence_length,
    image_base_dir,
    hidrograma_base_dir,
    x_min, x_max,
    ny, nx,
    load_hidrograma,
    interpolate_hidrograma,
    generate_dataset_with_mass_terms,
    eps=1e-8
):
    results = []

    for sid in scenario_ids:
        print(f"📊 Evaluando escenario {sid}...", end="\r")
        
        # 1. Cargar datos del escenario
        image_dir = os.path.join(image_base_dir, f"escenario_{sid}")
        hidrograma_file = os.path.join(hidrograma_base_dir, f"hidrograma_sintetico_{sid}.txt")
        hidrograma_raw = load_hidrograma(hidrograma_file)
        hidrograma_values = interpolate_hidrograma(hidrograma_raw, dt_target=dt_target)
        
        # Obtenemos los datos (X, Y, V_prev, V_next, Q_t, Q_next)
        out = generate_dataset_with_mass_terms(image_dir, hidrograma_values)
        X_loc, Y_true_flat = out[0], out[1]
        
        if X_loc.shape[0] == 0: continue

        # Normalización y ajuste de dimensiones
        Xn = (X_loc - x_min) / (x_max - x_min + eps)
        if Xn.ndim == 2: Xn = Xn[..., np.newaxis]

        # 2. Iterar sobre cada modelo cargado
        for model_name, model_obj in models_dict.items():
            # Predicción: el modelo devuelve [depth, mask]
            pred_depth, pred_mask = model_obj.predict(Xn, verbose=0)
            
            # --- Métricas de Profundidad ---
            # d_loss = MSE ponderado o MAE (según tu función depth_loss_flat_np)
            d_loss = depth_loss_flat_np(Y_true_flat, pred_depth, h0=h0)
            
            # --- Métricas de Máscara (CSI) ---
            true_mask = (Y_true_flat > h0).astype(np.float32)
            # Si el modelo no usa máscara (ablation), evaluamos igual para comparar
            pred_mask_binary = (pred_mask > 0.5).astype(np.float32)
            csi_val = csi_np(true_mask > 0.5, pred_mask_binary > 0.5)
            
            # --- Métricas de Física (Error de Volumen Relativo) ---
            # Calculamos el residuo de masa directamente para comparar
            v_hat = np.sum(pred_depth, axis=1) * dx * dy
            v_prev, v_next = out[2], out[3]
            q_t, q_next = out[4], out[5]
            
            # Residuo simple de continuidad
            mass_res = np.mean(np.abs(v_hat - v_prev - dt * q_t) / (nx*ny*dx*dy))

            results.append({
                "scenario_id": sid,
                "model_name": model_name,
                "depth_loss": float(d_loss),
                "csi": float(csi_val),
                "mass_residual": float(mass_res),
                "n_frames": int(X_loc.shape[0])
            })
            
    return pd.DataFrame(results)

In [14]:
# Definir diccionario con los 6 modelos cargados
models_to_evaluate = {
    "Baseline": model_baseline_loaded,
    "MoE_Only": model_moe_only_loaded,
    "Mask_Single": model_mask_single_loaded,
    "Mask_MoE": model_mask_moe_loaded,
    "PINN_Single": model_pinn_single_loaded,
    "PINN_MoE_Full": model_full_loaded
}

# Ejecutar el resumen
df_test_comparison = summarize_ablation_scenarios(
    scenario_ids=test_scenarios, # Usa tus IDs de test definidos antes
    models_dict=models_to_evaluate,
    dt_target=300,
    h0=H0_MASK,
    sequence_length=SEQUENCE_LENGTH,
    image_base_dir=image_base_dir,
    hidrograma_base_dir=hidrograma_base_dir,
    x_min=x_min, x_max=x_max,
    ny=ny, nx=nx,
    load_hidrograma=load_hidrograma,
    interpolate_hidrograma=interpolate_hidrograma,
    generate_dataset_with_mass_terms=generate_dataset_with_mass_terms
)

# 3. Mostrar resultados promedio por modelo
summary_table = df_test_comparison.groupby("model_name")[["depth_loss", "csi", "mass_residual"]].mean()
print("\n--- COMPARATIVA FINAL DE ABLACIÓN (Promedio Test Set) ---")
print(summary_table)

# Guardar a CSV para tu tesis
summary_table.to_csv("summary_ablation_test_metrics.csv")

📊 Evaluando escenario 40...
--- COMPARATIVA FINAL DE ABLACIÓN (Promedio Test Set) ---
               depth_loss       csi  mass_residual
model_name                                        
Baseline         0.019778  0.081199       0.021978
Mask_MoE         0.011876  0.755255       0.018675
Mask_Single      0.018339  0.722829       0.018983
MoE_Only         0.004382  0.080607       0.010219
PINN_MoE_Full    0.011964  0.762478       0.017087
PINN_Single      0.018578  0.081009       0.017955
